# 00 — Setup and sanity check

Gets the detector stack (HierarchicalDet: DiffusionDet + Detectron2, Swin-Large
backbone) running on this Kaggle notebook, and confirms the whole pipeline
works end to end on a *tiny* amount of data before you spend GPU hours on the
real training run in `01_train_baseline_detector.ipynb`.

**Status of what's below**: every step was verified locally (macOS arm64,
CPU/MPS, no GPU) in the project's development session — see `SETUP.md` in the
repo for the exact recipe this notebook follows. The install steps are
adapted here for Kaggle's environment (skip reinstalling torch — use the
image's preinstalled build). This notebook itself has not been run on actual
Kaggle hardware yet — if a cell fails, the error message plus `SETUP.md`
should get you unstuck fast; please update this notebook with what you find
so the next person doesn't hit the same thing.

**Before running**: in the Kaggle notebook settings, turn on GPU (T4 x2 or
P100) and internet access.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/christopherh-88/Carries-Confidence.git
%cd Carries-Confidence
!git log --oneline -5

## 2. Check what Kaggle already gives you

Do NOT `pip install torch`/`torchvision` — Kaggle's image ships a specific
torch+CUDA build already matched to the GPU driver. Reinstalling torch is the
#1 way to break the compiled ops later (see SETUP.md).

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
import sys
print("python:", sys.version)

## 3. Install core requirements (fast, no GPU needed)

In [ ]:
!pip install -q -r requirements-core.txt

## 4. Clone the HierarchicalDet baseline

In [ ]:
!bash scripts/clone_baseline.sh

## 5. Install detectron2

Confirmed recipe (see SETUP.md "Backbone weights -- a real bug in the
upstream config" section and the section above it for the full writeup).
`--no-build-isolation` is required: detectron2's setup.py does `import torch`
at build time, and pip's isolated build env can't see the notebook's torch
without this flag.

In [ ]:
!pip install -q ninja
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q timm scipy Pillow

## 6. Confirm the compiled ops load

If this cell fails with an ImportError on `_C`, STOP — nothing downstream
will work until this resolves. Check the torch/CUDA version printed in step 2
matches what detectron2 was built against (rerun step 5 if you changed
anything).

In [ ]:
import detectron2
print("detectron2 version:", detectron2.__version__)
from detectron2 import _C
print("compiled ops (_C) import: OK")

## 7. Import HierarchicalDet's code against the REAL detectron2

Two shadowing gotchas, both handled the same way: HierarchicalDet vendors its
own copies of `detectron2/` (no compiled ops) and `pycocotools/` (no compiled
`_mask`) at `external/HierarchicalDet/`. If that path gets added to
`sys.path` *before* the real packages are imported, Python resolves
`import detectron2` to the vendored, broken copy instead of erroring clearly.

Fix: import the real packages fully first (so they're cached in
`sys.modules`), THEN add the HierarchicalDet path.

In [ ]:
import sys
import pycocotools.mask, pycocotools.coco, pycocotools.cocoeval  # cache the real one first
import detectron2
from detectron2 import _C  # already confirmed above, just re-stating the order

sys.path.insert(0, "external/HierarchicalDet")
from hierarchialdet.config import add_diffusiondet_config
print("hierarchialdet imports OK, using the real detectron2")

## 8. Download and convert the Swin-Large backbone weights

**Read this before running**: the config's own weights filename
(`swin_base_patch4_window7_224_22k.pkl`) is misleading — it implies
Swin-Base, but the config's `MODEL.SWIN.SIZE` is `L-22k` (Swin-**Large**).
DiffusionDet's official release only ships Swin-Base weights under that
filename; using them here would silently fail to load ~90% of the backbone
(detectron2's checkpointer warns per-tensor but does not raise an error).

Confirmed fix: download the actual Swin-Large-22k classification checkpoint
from Microsoft's official release and convert it to detectron2's format.

In [ ]:
import os
os.makedirs("models_weights", exist_ok=True)

!curl -sL "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window7_224_22k.pth" \
  -o models_weights/swin_large_patch4_window7_224_22k_raw.pth

import torch, pickle
ckpt = torch.load("models_weights/swin_large_patch4_window7_224_22k_raw.pth", map_location="cpu", weights_only=False)
assert ckpt["model"]["patch_embed.proj.weight"].shape[0] == 192, "expected Swin-Large's 192-dim embedding, got something else -- did the download change?"
converted = {"model": ckpt["model"], "__author__": "third_party", "matching_heuristics": True}
with open("models_weights/swin_large_patch4_window7_224_22k.pkl", "wb") as f:
    pickle.dump(converted, f)
os.remove("models_weights/swin_large_patch4_window7_224_22k_raw.pth")  # save disk space, no longer needed
print("converted OK")

## 9. Get the DENTEX dataset

If you've already uploaded DENTEX as a Kaggle Dataset (see
`docs/phase2_data_notes.md` in the repo for why: the Hugging Face download
needs gated auth that doesn't translate well to a Kaggle notebook), attach it
via "Add Data" in the notebook sidebar and point `DATA_ROOT` below at
`/kaggle/input/<your-dataset-name>/...`. Otherwise, this cell downloads it
directly (needs `huggingface-cli login` credentials — set `HF_TOKEN` as a
Kaggle secret first).

In [ ]:
# Option A: already attached as a Kaggle Dataset -- just set the path:
DATA_ROOT = "/kaggle/input/dentex/DENTEX/training_data/quadrant-enumeration-disease"

# Option B: download directly (uncomment if you didn't attach a dataset).
# Needs: Kaggle notebook Settings -> Add-ons -> Secrets -> HF_TOKEN
# from kaggle_secrets import UserSecretsClient
# import os
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
# !pip install -q huggingface_hub
# !python scripts/download_dentex.py
# !unzip -q "data/dentex/DENTEX/training_data.zip" "training_data/quadrant-enumeration-disease/*" -d data/dentex/DENTEX/
# DATA_ROOT = "data/dentex/DENTEX/training_data/quadrant-enumeration-disease"

import os
assert os.path.exists(f"{DATA_ROOT}/xrays"), f"expected images at {DATA_ROOT}/xrays -- check DATA_ROOT above"
print("DATA_ROOT ok:", DATA_ROOT, "-", len(os.listdir(f"{DATA_ROOT}/xrays")), "images")

## 10. Build the real model and confirm the checkpoint loads cleanly

This is the check that catches the Swin-Base/Large mismatch bug from step 8
if it recurs. **Zero** "will not be loaded" lines should mention any
`backbone.bottom_up.*` key. Lines about `head.*`, FPN lateral/output convs,
or diffusion-schedule buffers (`alphas_cumprod` etc.) are expected and fine
— those aren't part of an ImageNet classification checkpoint.

In [ ]:
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer

cfg = get_cfg()
add_diffusiondet_config(cfg)
cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
cfg.MODEL.WEIGHTS = "models_weights/swin_large_patch4_window7_224_22k.pkl"
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = build_model(cfg)
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
n_params = sum(p.numel() for p in model.parameters())
print(f"model built OK: {type(model).__name__}, {n_params:,} params, device={cfg.MODEL.DEVICE}")
print()
print("^^ scroll up: confirm no 'will not be loaded' lines mention backbone.bottom_up.*")

## 11. Forward pass and one real training step (tiny sanity check)

Uses 2 real DENTEX images, not synthetic data — the actual target for
"confirm a baseline forward pass" before committing to the full run. Fast
regardless of GPU/CPU since it's only 2 images once.

In [ ]:
import copy
import cv2
import numpy as np
from detectron2.structures import Instances, Boxes

sys.path.insert(0, ".")  # repo root, for src.*
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2
from detectron2.data import DatasetCatalog

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
train_dicts = DatasetCatalog.get("custom_train_class")
print("registered custom_train_class:", len(train_dicts), "images")

def simple_mapper(d, target_size=800, device="cpu"):
    d = copy.deepcopy(d)
    img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
    h0, w0 = img.shape[:2]
    img = cv2.resize(img, (target_size, target_size))
    scale_x, scale_y = target_size / w0, target_size / h0

    inst = Instances((target_size, target_size))
    boxes, c1, c2, c3 = [], [], [], []
    for ann in d["annotations"]:
        x, y, w, h = ann["bbox"]
        boxes.append([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y])
        c1.append(ann["category_id_1"]); c2.append(ann["category_id_2"]); c3.append(ann["category_id_3"])
    inst.gt_boxes = Boxes(torch.tensor(boxes, dtype=torch.float32)) if boxes else Boxes(torch.zeros(0, 4))
    inst.gt_classes_1 = torch.tensor(c1, dtype=torch.int64)
    inst.gt_classes_2 = torch.tensor(c2, dtype=torch.int64)
    inst.gt_classes_3 = torch.tensor(c3, dtype=torch.int64)

    return {
        "image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)).to(device),
        "height": target_size, "width": target_size,
        "instances": inst.to(device),
    }

device = cfg.MODEL.DEVICE
batch = [simple_mapper(train_dicts[0], device=device), simple_mapper(train_dicts[1], device=device)]
print("batch built, boxes per image:", [len(b["instances"]) for b in batch])

model.eval()
with torch.no_grad():
    out = model(batch)
print("inference OK, instances predicted:", [len(o["instances"]) for o in out])

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
loss_dict = model(batch)
loss = sum(loss_dict.values())
loss.backward()
optimizer.step()
print("training step OK, losses:", {k: round(v.item(), 3) for k, v in list(loss_dict.items())[:5]})
print()
print("=== SETUP VERIFIED. Proceed to 01_train_baseline_detector.ipynb ===")